# SentimentScope — BERT-like (exact architecture + pretrained weights)

Same IMDB setup as the main notebook: `train/val` split with `random_state=42`, `bert-base-uncased` tokenizer, max length 256, no word-drop augmentation.

**Reproducibility:** `from_pretrained` pulls a fixed public checkpoint (deterministic once cached). Fine-tuning is still **stochastic** (shuffle, dropout). Tighter control: `torch.manual_seed(42)`, `torch.cuda.manual_seed_all(42)`, `torch.backends.cudnn.deterministic = True`, and a seeded `DataLoader` generator. **For reviewers:** run all cells, then save the notebook with outputs.


In [ ]:
import os
import math
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

BASE = os.getcwd()
train_pos_path = os.path.join(BASE, "aclImdb/train/pos")
train_neg_path = os.path.join(BASE, "aclImdb/train/neg")
test_pos_path = os.path.join(BASE, "aclImdb/test/pos")
test_neg_path = os.path.join(BASE, "aclImdb/test/neg")

for path in [train_pos_path, train_neg_path, test_pos_path, test_neg_path]:
    assert os.path.isdir(path), f"Path not found: {path} (run from SentimentScope_submission, extract aclImdb if needed)"
print("Dataset paths OK.")

def load_dataset(folder):
    texts = []
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                texts.append(f.read())
    return texts

train_pos = load_dataset(train_pos_path)
train_neg = load_dataset(train_neg_path)
test_pos = load_dataset(test_pos_path)
test_neg = load_dataset(test_neg_path)

train_df = pd.DataFrame({"review": train_pos + train_neg, "label": [1] * len(train_pos) + [0] * len(train_neg)})
test_df = pd.DataFrame({"review": test_pos + test_neg, "label": [1] * len(test_pos) + [0] * len(test_neg)})
print(f"Train: {len(train_df)} | Test: {len(test_df)}")

train_size = int(0.9 * len(train_df))
shuffled_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_data = shuffled_df.iloc[:train_size]
val_data = shuffled_df.iloc[train_size:]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LENGTH = 256
BATCH_SIZE = 32

def trim_middle_to_fit(text, tokenizer, max_length):
    words = text.split()
    if len(words) <= 1:
        return text
    n_tokens = len(tokenizer.encode(text, add_special_tokens=True))
    if n_tokens <= max_length:
        return text
    target_words = max(1, int(len(words) * max_length / n_tokens * 0.95))
    head_len = target_words // 2
    tail_len = target_words - head_len
    if head_len + tail_len >= len(words):
        return text
    return " ".join(words[:head_len] + words[-tail_len:])

class IMDBDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=MAX_LENGTH):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]["review"]
        label = self.data.iloc[idx]["label"]
        if len(self.tokenizer.encode(text, add_special_tokens=True)) > self.max_length:
            text = trim_middle_to_fit(text, self.tokenizer, self.max_length)
        tokens = self.tokenizer(
            text, truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt",
        )
        return tokens["input_ids"].squeeze(0), tokens["attention_mask"].squeeze(0), label

train_dataset = IMDBDataset(train_data, tokenizer)
val_dataset = IMDBDataset(val_data, tokenizer)
test_dataset = IMDBDataset(test_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


Model matches **bert-base-uncased**: post-LN blocks, `[CLS]` + pooler + tanh, token-type embeddings, embedding LayerNorm + dropout. Pretrained weights are mapped from `BertModel` into this implementation.


In [ ]:
config = {
    "vocabulary_size": tokenizer.vocab_size,
    "num_classes": 2,
    "d_embed": 768,
    "context_size": MAX_LENGTH,
    "layers_num": 12,
    "heads_num": 12,
    "head_size": 64,
    "dropout_rate": 0.1,
    "use_bias": True,
}

class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.Q_weights = nn.Linear(config["d_embed"], config["head_size"], bias=config["use_bias"])
        self.K_weights = nn.Linear(config["d_embed"], config["head_size"], bias=config["use_bias"])
        self.V_weights = nn.Linear(config["d_embed"], config["head_size"], bias=config["use_bias"])
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input, padding_mask):
        Q, K, V = self.Q_weights(input), self.K_weights(input), self.V_weights(input)
        scores = (Q @ K.transpose(1, 2)) / math.sqrt(K.shape[-1])
        scores = scores.masked_fill(padding_mask == 0, float("-inf"))
        scores = self.dropout(torch.softmax(scores, dim=-1))
        return scores @ V

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList([AttentionHead(config) for _ in range(config["heads_num"])])
        self.linear = nn.Linear(config["heads_num"] * config["head_size"], config["d_embed"])
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input, padding_mask):
        x = torch.cat([head(input, padding_mask) for head in self.heads], dim=-1)
        return self.dropout(self.linear(x))

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.linear_layers = nn.Sequential(
            nn.Linear(config["d_embed"], 4 * config["d_embed"]),
            nn.GELU(),
            nn.Linear(4 * config["d_embed"], config["d_embed"]),
            nn.Dropout(config["dropout_rate"]),
        )

    def forward(self, input):
        return self.linear_layers(input)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.multi_head = MultiHeadAttention(config)
        self.layer_norm_1 = nn.LayerNorm(config["d_embed"], eps=1e-12)
        self.feed_forward = FeedForward(config)
        self.layer_norm_2 = nn.LayerNorm(config["d_embed"], eps=1e-12)

    def forward(self, input, padding_mask):
        x = self.layer_norm_1(input + self.multi_head(input, padding_mask))
        return self.layer_norm_2(x + self.feed_forward(x))

class DemoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        d = config["d_embed"]
        self.token_embedding_layer = nn.Embedding(config["vocabulary_size"], d)
        self.positional_embedding_layer = nn.Embedding(config["context_size"], d)
        self.token_type_embedding_layer = nn.Embedding(2, d)
        self.embed_layer_norm = nn.LayerNorm(d, eps=1e-12)
        self.embed_dropout = nn.Dropout(config["dropout_rate"])
        self.blocks = nn.ModuleList([Block(config) for _ in range(config["layers_num"])])
        self.pooler_dense = nn.Linear(d, d)
        self.pooler_activation = nn.Tanh()
        self.pooler_dropout = nn.Dropout(config["dropout_rate"])
        self.classifier = nn.Linear(d, config["num_classes"])

    def forward(self, token_ids, attention_mask):
        _, tokens_num = token_ids.shape
        positions = torch.arange(tokens_num, device=token_ids.device)
        token_type_ids = torch.zeros_like(token_ids)
        x = (
            self.token_embedding_layer(token_ids)
            + self.positional_embedding_layer(positions).unsqueeze(0)
            + self.token_type_embedding_layer(token_type_ids)
        )
        x = self.embed_dropout(self.embed_layer_norm(x))
        padding_mask = attention_mask.unsqueeze(1).expand(-1, tokens_num, -1)
        for block in self.blocks:
            x = block(x, padding_mask)
        cls_output = x[:, 0]
        pooled = self.pooler_dropout(self.pooler_activation(self.pooler_dense(cls_output)))
        return self.classifier(pooled)


In [ ]:
def calculate_accuracy(model, data_loader, device):
    model.eval()
    total_correct = total_samples = 0
    with torch.no_grad():
        for input_ids, attention_mask, labels in data_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    return (total_correct / total_samples) * 100

def load_bert_pretrained(model, config):
    from transformers import BertModel
    bert = BertModel.from_pretrained("bert-base-uncased")
    b = bert.state_dict()
    num_heads, head_size = config["heads_num"], config["head_size"]
    new_state = {}
    new_state["token_embedding_layer.weight"] = b["embeddings.word_embeddings.weight"]
    pos_emb = b["embeddings.position_embeddings.weight"]
    new_state["positional_embedding_layer.weight"] = pos_emb[: config["context_size"]]
    new_state["token_type_embedding_layer.weight"] = b["embeddings.token_type_embeddings.weight"]
    new_state["embed_layer_norm.weight"] = b["embeddings.LayerNorm.weight"]
    new_state["embed_layer_norm.bias"] = b["embeddings.LayerNorm.bias"]
    for i in range(config["layers_num"]):
        prefix = f"encoder.layer.{i}"
        q_w = b[f"{prefix}.attention.self.query.weight"].chunk(num_heads, dim=0)
        q_b = b[f"{prefix}.attention.self.query.bias"].chunk(num_heads, dim=0)
        k_w = b[f"{prefix}.attention.self.key.weight"].chunk(num_heads, dim=0)
        k_b = b[f"{prefix}.attention.self.key.bias"].chunk(num_heads, dim=0)
        v_w = b[f"{prefix}.attention.self.value.weight"].chunk(num_heads, dim=0)
        v_b = b[f"{prefix}.attention.self.value.bias"].chunk(num_heads, dim=0)
        for j in range(num_heads):
            h = f"blocks.{i}.multi_head.heads.{j}"
            new_state[f"{h}.Q_weights.weight"] = q_w[j]
            new_state[f"{h}.Q_weights.bias"] = q_b[j]
            new_state[f"{h}.K_weights.weight"] = k_w[j]
            new_state[f"{h}.K_weights.bias"] = k_b[j]
            new_state[f"{h}.V_weights.weight"] = v_w[j]
            new_state[f"{h}.V_weights.bias"] = v_b[j]
        new_state[f"blocks.{i}.multi_head.linear.weight"] = b[f"{prefix}.attention.output.dense.weight"]
        new_state[f"blocks.{i}.multi_head.linear.bias"] = b[f"{prefix}.attention.output.dense.bias"]
        new_state[f"blocks.{i}.layer_norm_1.weight"] = b[f"{prefix}.attention.output.LayerNorm.weight"]
        new_state[f"blocks.{i}.layer_norm_1.bias"] = b[f"{prefix}.attention.output.LayerNorm.bias"]
        new_state[f"blocks.{i}.feed_forward.linear_layers.0.weight"] = b[f"{prefix}.intermediate.dense.weight"]
        new_state[f"blocks.{i}.feed_forward.linear_layers.0.bias"] = b[f"{prefix}.intermediate.dense.bias"]
        new_state[f"blocks.{i}.feed_forward.linear_layers.2.weight"] = b[f"{prefix}.output.dense.weight"]
        new_state[f"blocks.{i}.feed_forward.linear_layers.2.bias"] = b[f"{prefix}.output.dense.bias"]
        new_state[f"blocks.{i}.layer_norm_2.weight"] = b[f"{prefix}.output.LayerNorm.weight"]
        new_state[f"blocks.{i}.layer_norm_2.bias"] = b[f"{prefix}.output.LayerNorm.bias"]
    new_state["pooler_dense.weight"] = b["pooler.dense.weight"]
    new_state["pooler_dense.bias"] = b["pooler.dense.bias"]
    missing, unexpected = model.load_state_dict(new_state, strict=False)
    print("Loaded BERT weights. Missing (random init):", missing)
    if unexpected:
        print("Unexpected keys:", unexpected)
    del bert
    return model


**Training:** `USE_PRETRAINED=True` → fine-tune with AdamW `lr=2e-5`, 10% warmup, linear decay, 3 epochs (classifier head is randomly initialized). Set `False` for from-scratch training (higher LR, more epochs) as in the `.py` script.


In [ ]:
USE_PRETRAINED = True
LR = 2e-5 if USE_PRETRAINED else 5e-4
WEIGHT_DECAY = 0.01
EPOCHS = 3 if USE_PRETRAINED else 25
LOAD_CHECKPOINT = False

checkpoint_path = os.path.join(BASE, "demogpt_bertlike.pt")

model = DemoGPT(config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

if LOAD_CHECKPOINT and os.path.isfile(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
    print("Resumed from", checkpoint_path)
elif USE_PRETRAINED:
    model = load_bert_pretrained(model, config)
    print("Fine-tuning BERT pretrained encoder + new classifier.")

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
steps_per_epoch = len(train_loader)
total_steps = EPOCHS * steps_per_epoch
warmup_steps = int(0.1 * total_steps) if USE_PRETRAINED else steps_per_epoch * 2

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return current_step / max(1, warmup_steps)
    progress = (current_step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 1.0 - progress

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
mode = "fine-tuning (pretrained)" if USE_PRETRAINED else "from scratch"
print(f"{mode} | lr={LR} | epochs={EPOCHS} | checkpoint: {os.path.basename(checkpoint_path)}")


In [ ]:
best_val = 0.0
best_epoch = 0
if LOAD_CHECKPOINT and os.path.isfile(checkpoint_path):
    best_val = calculate_accuracy(model, val_loader, device)
    print(f"Checkpoint val acc: {best_val:.2f}%\n")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for step, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{EPOCHS}] Step [{step+1}/{len(train_loader)}] "
                f"Loss {running_loss/100:.4f} LR {scheduler.get_last_lr()[0]:.2e}"
            )
            running_loss = 0.0
    val_accuracy = calculate_accuracy(model, val_loader, device)
    print(f"Epoch {epoch+1} — val accuracy: {val_accuracy:.2f}%")
    if val_accuracy > best_val:
        best_val, best_epoch = val_accuracy, epoch + 1
        torch.save(model.state_dict(), checkpoint_path)

print(f"\nBest validation: {best_val:.2f}% (epoch {best_epoch})")


In [ ]:
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
test_accuracy = calculate_accuracy(model, test_loader, device)
print(f"Test accuracy (best val checkpoint): {test_accuracy:.2f}%")
print(f"Summary: val {best_val:.2f}% (epoch {best_epoch}) → test {test_accuracy:.2f}%")


Best weights are saved as `demogpt_bertlike.pt` in this folder.
